# 02 — Robot Data Quality
Quality flags and safe split-gap repair.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DataConfig
from src.data_quality import run_quality_pipeline


In [ ]:
raw_stocks = pd.read_parquet(
    PROJECT_ROOT / "data" / "raw" / "bist100_robot_raw.parquet"
)
raw_market = pd.read_parquet(
    PROJECT_ROOT / "data" / "raw" / "xu100_robot_raw.parquet"
)

config = DataConfig()
stock_quality = run_quality_pipeline(raw_stocks, config, apply_split_repairs=True)
market_quality = run_quality_pipeline(raw_market, config, apply_split_repairs=True)

display(stock_quality.summary.head(20))
display(stock_quality.split_repairs)


In [ ]:
processed_dir = PROJECT_ROOT / "data" / "processed"
results_dir = PROJECT_ROOT / "results"
processed_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

stock_quality.enriched.to_parquet(
    processed_dir / "bist100_robot_enriched.parquet", index=False
)
stock_quality.clean.to_parquet(
    processed_dir / "bist100_robot_clean.parquet", index=False
)
market_quality.clean.to_parquet(
    processed_dir / "xu100_robot_clean.parquet", index=False
)
stock_quality.summary.to_csv(
    results_dir / "data_quality_summary.csv", index=False
)
stock_quality.split_repairs.to_csv(
    results_dir / "split_repair_log.csv", index=False
)
